# 01 · Data Cleaning & Feature Engineering Pipeline

**Input :** `../data/Train Dataset.csv`  
**Output:** `../data/combined_dataset.csv` (SMOTE-balanced train + original test split)

Run all cells top-to-bottom before opening `02_model_pipeline.ipynb`.

## 1. Imports

In [ ]:
import warnings
warnings.filterwarnings('ignore')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from IPython.display import display

from sklearn.model_selection import train_test_split
from sklearn.ensemble import RandomForestClassifier
from sklearn.metrics import accuracy_score
from imblearn.over_sampling import SMOTE

sns.set_theme(style='whitegrid')
RANDOM_STATE = 42
print("All imports OK")

## 2. Load Raw Training Dataset

In [ ]:
df = pd.read_csv("../data/Train Dataset.csv")
print("Raw shape:", df.shape)
display(df.head())

## 3. Standardise Column Names

Convert all column names to lowercase with underscores.

In [ ]:
df.columns = df.columns.str.strip().str.lower().str.replace(' ', '_')
print("Columns:", df.columns.tolist())

## 4. Drop Artefact Columns

Remove auto-generated Pandas index column (`Unnamed: 0`) if present.

In [ ]:
if 'unnamed:_0' in df.columns:
    df = df.drop(columns=['unnamed:_0'])
print("Columns after drop:", df.columns.tolist())

## 5. Remove Exact Duplicate Rows

In [ ]:
before = df.shape[0]
df = df.drop_duplicates().reset_index(drop=True)
print("Rows removed (exact duplicates):", before - df.shape[0])
print("Remaining rows:", df.shape[0])

## 6. Resolve Feature–Label Conflicts

Remove rows where identical feature vectors map to **different** crop labels (noisy / mislabelled data).

In [ ]:
features = df.drop(columns=['crop'])
dup_mask = features.duplicated(keep=False)
dup_df   = df[dup_mask]

clean_df    = dup_df.groupby(list(features.columns)).filter(lambda x: x['crop'].nunique() == 1)
non_dup_df  = df[~dup_mask]
df = pd.concat([clean_df, non_dup_df]).drop_duplicates().reset_index(drop=True)
print("Shape after conflict removal:", df.shape)

## 7. Handle Missing Values

Fill missing numeric values with column medians.

In [ ]:
null_counts = df.isnull().sum()
print("Null counts:")
print(null_counts[null_counts > 0] if null_counts.any() else 'No nulls found')

for col in df.select_dtypes(include='number').columns:
    df[col] = df[col].fillna(df[col].median())
print("Missing values handled.")

## 8. Filter Physically Impossible Values

Remove negative nutrients and pH outside [0, 14].

In [ ]:
before = df.shape[0]
for col in ['n', 'p', 'k', 'rainfall']:
    if col in df.columns:
        df = df[df[col] >= 0]
if 'ph' in df.columns:
    df = df[(df['ph'] >= 0) & (df['ph'] <= 14)]
df = df.reset_index(drop=True)
print("Rows removed (impossible values):", before - df.shape[0])

## 9. Clip Outliers (1.5 × IQR Winsorization)

In [ ]:
def clip_outliers(df, cols):
    for col in cols:
        q1, q3 = df[col].quantile([0.25, 0.75])
        iqr = q3 - q1
        df[col] = df[col].clip(q1 - 1.5 * iqr, q3 + 1.5 * iqr)
    return df

num_cols = df.select_dtypes(include='number').columns.tolist()
df = clip_outliers(df.copy(), num_cols)
print("Outlier clipping applied to", len(num_cols), "numeric columns.")

## 10. Crop Class Distribution (Before Balancing)

In [ ]:
print(df['crop'].value_counts())

fig, ax = plt.subplots(figsize=(15, 5))
counts = df['crop'].value_counts()
sns.barplot(x=counts.index, y=counts.values, palette='viridis', ax=ax)
ax.set_title('Pre-SMOTE Class Distribution', fontsize=14, fontweight='bold')
ax.set_xlabel('Crop'); ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

## 11. Feature Engineering

| Feature | Formula | Agronomic Meaning |
|---|---|---|
| `n_p_ratio` | N / (P + 1) | Nitrogen–Phosphorus balance |
| `n_k_ratio` | N / (K + 1) | Nitrogen–Potassium balance |
| `temp_rainfall` | temperature × rainfall | Heat–moisture interaction |

In [ ]:
df['n_p_ratio']     = df['n'] / (df['p'] + 1)
df['n_k_ratio']     = df['n'] / (df['k'] + 1)
df['temp_rainfall'] = df['temperature'] * df['rainfall']
print("Features after engineering:", df.columns.tolist())
display(df.head(3))

## 12. Stratified Train / Test Split

`stratify=y` preserves class proportions in both splits.

In [ ]:
X = df.drop(columns=['crop'])
y = df['crop']

X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, stratify=y, random_state=RANDOM_STATE
)
print("Train size :", X_train.shape[0], "rows")
print("Test size  :", X_test.shape[0],  "rows")
print("Features   :", X_train.shape[1])

## 13. SMOTE Oversampling (Training Set Only)

> ⚠️ SMOTE is applied **only** to `X_train`/`y_train`. The test set remains untouched to prevent data leakage.

In [ ]:
smote = SMOTE(random_state=RANDOM_STATE)
X_train_smote, y_train_smote = smote.fit_resample(X_train, y_train)

print("X_train before SMOTE:", X_train.shape[0], "rows")
print("X_train after  SMOTE:", X_train_smote.shape[0], "rows")

fig, ax = plt.subplots(figsize=(15, 5))
smote_counts = pd.Series(y_train_smote).value_counts()
sns.barplot(x=smote_counts.index, y=smote_counts.values, palette='rocket', ax=ax)
ax.set_title('Post-SMOTE Training Class Distribution (Balanced)', fontsize=14, fontweight='bold')
ax.set_xlabel('Crop'); ax.set_ylabel('Count')
plt.xticks(rotation=45, ha='right'); plt.tight_layout(); plt.show()

## 14. Sanity Check — Quick Random Forest Baseline

A fast RF trained on the SMOTE data verifies the pipeline is correct. Expect >95%.

In [ ]:
X_tr, X_val, y_tr, y_val = train_test_split(
    X_train_smote, y_train_smote, test_size=0.2,
    stratify=y_train_smote, random_state=RANDOM_STATE
)
rf_check = RandomForestClassifier(n_estimators=50, random_state=RANDOM_STATE, n_jobs=-1)
rf_check.fit(X_tr, y_tr)
val_acc = accuracy_score(y_val, rf_check.predict(X_val))
print("Sanity-check RF accuracy:", round(val_acc, 4))
print("Pipeline OK" if val_acc > 0.90 else "WARNING: low accuracy — review cleaning steps")

## 15. Build & Save `combined_dataset.csv`

**Structure:** SMOTE-resampled training rows + original (unmodified) test rows.

This is the exact dataset consumed by `02_model_pipeline.ipynb`.

In [ ]:
# SMOTE-resampled train portion
df_train = pd.DataFrame(X_train_smote, columns=X.columns)
df_train['crop']  = y_train_smote.values if hasattr(y_train_smote, 'values') else y_train_smote
df_train['split'] = 'train'

# Original test portion (no leakage)
df_test = X_test.copy()
df_test['crop']  = y_test.values
df_test['split'] = 'test'

combined = pd.concat([df_train, df_test], ignore_index=True)

output_path = '../data/combined_dataset.csv'
combined.to_csv(output_path, index=False)

print("Saved  ->", output_path)
print("Shape  :", combined.shape)
print("Columns:", combined.columns.tolist())
display(combined.head(3))

## 16. Final Dataset Summary

In [ ]:
print("=== combined_dataset.csv summary ===")
print("Total rows   :", combined.shape[0])
print("Train rows   :", combined[combined["split"]=="train"].shape[0])
print("Test rows    :", combined[combined["split"]=="test"].shape[0])
print("Crop classes :", combined["crop"].nunique())
print("Columns      :", combined.columns.tolist())